# Experiment 3: ONNX Export, INT8 Quantization & Accuracy Evaluation

**Goal:** Export trained models to ONNX, quantize to INT8, and measure accuracy before/after quantization.

**Pipeline per model:**
1. Export trained weights → FP32 ONNX
2. Verify FP32 ONNX accuracy matches PyTorch (sanity check)
3. Quantize FP32 ONNX → INT8 ONNX
4. Evaluate INT8 accuracy (measure quantization drop)
5. Record file sizes

**Models:**
- YOLO11n (Ultralytics)
- YOLOv10n (Ultralytics)
- YOLO26n (Ultralytics)

All accuracy numbers are final — the Pi 5 only adds latency/FPS.

In [1]:
!pip install onnx onnxruntime-gpu

---
## Configuration

In [3]:
import os
import shutil

# =============================================================
# PATHS — UPDATE THESE IF YOUR RUNS HAVE DIFFERENT NAMES
# =============================================================

# Trained model weights from Experiment 2
YOLO11N_PT  = "/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/real_unfrozen_e402/weights/best.pt"
YOLOV10N_PT = "/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolov10n_real_e402/weights/best.pt"
YOLO26N_PT  = "/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolo26n_real_e403/weights/best.pt"

# Output directory for all exported/quantized models
OUTPUT_DIR = "/workspace/FYP_RESULTS/Final-Year-Project/runs/quantized"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# YOLO data config (for Ultralytics val)
YOLO_DATA = "mydata_capped.yaml"

# Verify paths exist
for name, path in [("YOLO11n weights", YOLO11N_PT), ("YOLOv10n weights", YOLOV10N_PT),
                   ("YOLO26n weights", YOLO26N_PT)]:
    if os.path.exists(path):
        print(f"  {name}: {path}")
    else:
        print(f"  WARNING — {name} NOT FOUND: {path}")

  YOLO11n weights: /workspace/FYP_RESULTS/Final-Year-Project/runs/detect/real_unfrozen_e402/weights/best.pt
  YOLOv10n weights: /workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolov10n_real_e402/weights/best.pt
  YOLO26n weights: /workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolo26n_real_e403/weights/best.pt


---
## Part 1a: YOLO26n — Export & Quantize

Ultralytics makes this straightforward:
- `model.export()` for ONNX
- `model.val()` to evaluate any format (`.pt`, `.onnx`, etc.)

### 1a. Export to FP32 ONNX + Verify Accuracy

In [3]:
from ultralytics import YOLO

# Load trained PyTorch model
model = YOLO(YOLO26N_PT)

# Export to FP32 ONNX (end2end=False excludes NMS from the graph,
# which makes the model quantization-friendly. Ultralytics still
# applies NMS automatically during val() on the Python side.)
onnx_path = model.export(format="onnx", imgsz=832, end2end=False)

# Copy to output directory with clean name
yolo_fp32_path = os.path.join(OUTPUT_DIR, "yolo26n_fp32.onnx")
shutil.copy2(onnx_path, yolo_fp32_path)
print(f"\nFP32 ONNX saved to: {yolo_fp32_path}")
print(f"Size: {os.path.getsize(yolo_fp32_path) / (1024*1024):.1f} MB")

Ultralytics 8.4.16 🚀 Python-3.12.3 torch-2.9.1+cu128 CPU (Intel Xeon Gold 6133 CPU @ 2.50GHz)
YOLO26n summary (fused): 146 layers, 2,494,694 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from '/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolo26n_real_e403/weights/best.pt' with input shape (1, 3, 832, 832) BCHW and output shape(s) (1, 5, 14196) (5.1 MB)

ONNX: starting export with onnx 1.19.1 opset 22...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1447: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.85...
ONNX: export success ✅ 1.9s, saved as '/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolo26n_real_e403/weights/best.onnx' (9.5 MB)

Export complete (2.4s)
Results saved to /workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolo26n_real_e403/weights
Predict:         yolo predict task=detect model=/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolo26n_real_e403/weights/best.onnx imgsz=832 
Validate:        yolo val task=detect model=/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolo26n_real_e403/weights/best.onnx imgsz=832 data=mydata_capped.yaml  
Visualize:       https://netron.app

FP32 ONNX saved to: /workspace/FYP_RESULTS/Final-Year-Project/runs/quantized/yolo26n_fp32.onnx
Size: 9.5 MB


In [4]:
# Verify FP32 ONNX accuracy matches PyTorch
onnx_model = YOLO(yolo_fp32_path)
metrics_fp32 = onnx_model.val(data=YOLO_DATA, split="test", imgsz=832)

print(f"\n===== YOLO26n FP32 ONNX — TEST METRICS =====")
print(f"mAP@0.50:        {metrics_fp32.box.map50:.4f}")
print(f"mAP@0.50:0.95:   {metrics_fp32.box.map:.4f}")
print(f"Precision:        {metrics_fp32.box.mp:.4f}")
print(f"Recall:           {metrics_fp32.box.mr:.4f}")
print(f"==============================================")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Ultralytics 8.4.16 🚀 Python-3.12.3 torch-2.9.1+cu128 CPU (Intel Xeon Gold 6133 CPU @ 2.50GHz)
Loading /workspace/FYP_RESULTS/Final-Year-Project/runs/quantized/yolo26n_fp32.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.2 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 832, 832)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 173.3±98.7 MB/s, size: 230.4 KB)
val: Scanning /workspace/FYP_RESULTS/Final-Year-Project/datasets/my_data/labels.cache... 660 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 660/660 102.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 660/660 4.6it/s 2:22<0.2s
                   all        660        793      0.957      0.874      0.923      0.742
Speed: 2.3ms preprocess, 159.8ms inference

### 1b. Quantize to INT8 (Static Quantization)

**Why static, not dynamic?** `quantize_dynamic` only quantizes MatMul/Linear layers.
YOLO is mostly Conv layers, so dynamic quantization would barely change anything.

Static quantization calibrates **all** layers (including Conv) using real images,
producing a properly quantized INT8 model with real size reduction.

In [5]:
import numpy as np
import onnxruntime as ort
import onnx
from onnx import shape_inference
from PIL import Image
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType

# -------------------------------------------------------
# 1. Calibration data reader
# -------------------------------------------------------

class ImageCalibrationReader(CalibrationDataReader):
    """Reads images and preprocesses them for ONNX model calibration."""

    def __init__(self, image_paths, input_name, imgsz=832):
        self.images = image_paths
        self.input_name = input_name
        self.imgsz = imgsz
        self.idx = 0

    def get_next(self):
        if self.idx >= len(self.images):
            return None

        img = Image.open(self.images[self.idx]).convert("RGB")
        img = img.resize((self.imgsz, self.imgsz))
        img_array = np.array(img, dtype=np.float32) / 255.0
        img_tensor = np.transpose(img_array, (2, 0, 1))[np.newaxis, ...]

        self.idx += 1
        return {self.input_name: img_tensor}

# -------------------------------------------------------
# 2. Load 100 calibration images from training set
# -------------------------------------------------------

with open("/workspace/train_capped.txt") as f:
    all_train_images = [line.strip() for line in f if line.strip()]
calib_images = all_train_images[:100]
print(f"Using {len(calib_images)} calibration images")

# -------------------------------------------------------
# 3. Shape inference
# -------------------------------------------------------

yolo_fp32_prep = yolo_fp32_path.replace(".onnx", "_prep.onnx")
model_onnx = onnx.load(yolo_fp32_path)
model_onnx = shape_inference.infer_shapes(model_onnx)
onnx.save(model_onnx, yolo_fp32_prep)
print("Shape inference done")

# -------------------------------------------------------
# 4. Find detection head nodes to EXCLUDE from quantization.
#    Only the backbone+neck get INT8; the head stays FP32
#    to preserve sensitive score/box regression precision.
# -------------------------------------------------------

head_nodes = []
for node in model_onnx.graph.node:
    # YOLO26 detection head layers (model.22, model.23)
    if any(tag in node.name for tag in ["model.22", "model.23"]):
        head_nodes.append(node.name)
print(f"Excluding {len(head_nodes)} detection head nodes from quantization")

# -------------------------------------------------------
# 5. Static quantization — backbone+neck only
# -------------------------------------------------------

sess = ort.InferenceSession(yolo_fp32_prep)
input_name = sess.get_inputs()[0].name

reader = ImageCalibrationReader(calib_images, input_name, imgsz=832)

yolo_int8_path = os.path.join(OUTPUT_DIR, "yolo26n_int8.onnx")

quantize_static(
    yolo_fp32_prep,
    yolo_int8_path,
    calibration_data_reader=reader,
    quant_format=QuantFormat.QDQ,
    per_channel=True,
    weight_type=QuantType.QInt8,
    activation_type=QuantType.QUInt8,
    nodes_to_exclude=head_nodes,
)

print(f"\nINT8 ONNX saved to: {yolo_int8_path}")
print(f"FP32 size: {os.path.getsize(yolo_fp32_path) / (1024*1024):.1f} MB")
print(f"INT8 size: {os.path.getsize(yolo_int8_path) / (1024*1024):.1f} MB")

# Clean up preprocessed file
os.remove(yolo_fp32_prep)

Using 100 calibration images
Shape inference done
Excluding 114 detection head nodes from quantization



INT8 ONNX saved to: /workspace/FYP_RESULTS/Final-Year-Project/runs/quantized/yolo26n_int8.onnx
FP32 size: 9.5 MB
INT8 size: 4.6 MB


In [6]:
# Evaluate INT8 accuracy
int8_model = YOLO(yolo_int8_path)
metrics_int8 = int8_model.val(data=YOLO_DATA, split="test", imgsz=832)

print(f"\n===== YOLO26n INT8 ONNX — TEST METRICS =====")
print(f"mAP@0.50:        {metrics_int8.box.map50:.4f}")
print(f"mAP@0.50:0.95:   {metrics_int8.box.map:.4f}")
print(f"Precision:        {metrics_int8.box.mp:.4f}")
print(f"Recall:           {metrics_int8.box.mr:.4f}")
print(f"==============================================")

# Show accuracy drop
print(f"\n--- Quantization Impact ---")
print(f"mAP@50-95 drop:  {metrics_fp32.box.map - metrics_int8.box.map:+.4f}")
print(f"Precision drop:  {metrics_fp32.box.mp - metrics_int8.box.mp:+.4f}")
print(f"Recall drop:     {metrics_fp32.box.mr - metrics_int8.box.mr:+.4f}")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Ultralytics 8.4.16 🚀 Python-3.12.3 torch-2.9.1+cu128 CPU (Intel Xeon Gold 6133 CPU @ 2.50GHz)
Loading /workspace/FYP_RESULTS/Final-Year-Project/runs/quantized/yolo26n_int8.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.2 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 832, 832)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1843.2±447.3 MB/s, size: 232.2 KB)
val: Scanning /workspace/FYP_RESULTS/Final-Year-Project/datasets/my_data/labels.cache... 660 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 660/660 184.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 660/660 5.0it/s 2:13<0.2s
                   all        660        793      0.942      0.873      0.907      0.712
Speed: 1.6ms preprocess, 141.7ms inferen

---
## Part 1b: YOLO11n — Export & Quantize

In [7]:
from ultralytics import YOLO

# Load trained YOLO11n PyTorch model
model_11n = YOLO(YOLO11N_PT)

# Export to FP32 ONNX (end2end=False for quantization-friendly graph)
onnx_path = model_11n.export(format="onnx", imgsz=832, end2end=False)

# Copy to output directory with clean name
yolo11n_fp32_path = os.path.join(OUTPUT_DIR, "yolo11n_fp32.onnx")
shutil.copy2(onnx_path, yolo11n_fp32_path)
print(f"\nFP32 ONNX saved to: {yolo11n_fp32_path}")
print(f"Size: {os.path.getsize(yolo11n_fp32_path) / (1024*1024):.1f} MB")

Ultralytics 8.4.16 🚀 Python-3.12.3 torch-2.9.1+cu128 CPU (Intel Xeon Gold 6133 CPU @ 2.50GHz)
YOLO11n summary (fused): 100 layers, 2,582,347 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/real_unfrozen_e402/weights/best.pt' with input shape (1, 3, 832, 832) BCHW and output shape(s) (1, 5, 14196) (5.2 MB)

ONNX: starting export with onnx 1.19.1 opset 22...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1447: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.85...
ONNX: export success ✅ 1.3s, saved as '/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/real_unfrozen_e402/weights/best.onnx' (10.2 MB)

Export complete (1.7s)
Results saved to /workspace/FYP_RESULTS/Final-Year-Project/runs/detect/real_unfrozen_e402/weights
Predict:         yolo predict task=detect model=/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/real_unfrozen_e402/weights/best.onnx imgsz=832 
Validate:        yolo val task=detect model=/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/real_unfrozen_e402/weights/best.onnx imgsz=832 data=mydata_capped.yaml  
Visualize:       https://netron.app

FP32 ONNX saved to: /workspace/FYP_RESULTS/Final-Year-Project/runs/quantized/yolo11n_fp32.onnx
Size: 10.2 MB


In [8]:
# Verify FP32 ONNX accuracy matches PyTorch
onnx_model_11n = YOLO(yolo11n_fp32_path)
metrics_11n_fp32 = onnx_model_11n.val(data=YOLO_DATA, split="test", imgsz=832)

print(f"\n===== YOLO11n FP32 ONNX — TEST METRICS =====")
print(f"mAP@0.50:        {metrics_11n_fp32.box.map50:.4f}")
print(f"mAP@0.50:0.95:   {metrics_11n_fp32.box.map:.4f}")
print(f"Precision:        {metrics_11n_fp32.box.mp:.4f}")
print(f"Recall:           {metrics_11n_fp32.box.mr:.4f}")
print(f"==============================================")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Ultralytics 8.4.16 🚀 Python-3.12.3 torch-2.9.1+cu128 CPU (Intel Xeon Gold 6133 CPU @ 2.50GHz)
Loading /workspace/FYP_RESULTS/Final-Year-Project/runs/quantized/yolo11n_fp32.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.2 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 832, 832)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2266.2±243.3 MB/s, size: 250.5 KB)
val: Scanning /workspace/FYP_RESULTS/Final-Year-Project/datasets/my_data/labels.cache... 660 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 660/660 173.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 660/660 3.6it/s 3:04<0.3s
                   all        660        793      0.951      0.885      0.921      0.723
Speed: 3.0ms preprocess, 233.1ms inferen

In [9]:
import numpy as np
import re
import onnxruntime as ort
import onnx
from onnx import shape_inference
from PIL import Image
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType

class ImageCalibrationReader(CalibrationDataReader):
    def __init__(self, image_paths, input_name, imgsz=832):
        self.images = image_paths
        self.input_name = input_name
        self.imgsz = imgsz
        self.idx = 0
    def get_next(self):
        if self.idx >= len(self.images):
            return None
        img = Image.open(self.images[self.idx]).convert("RGB")
        img = img.resize((self.imgsz, self.imgsz))
        img_array = np.array(img, dtype=np.float32) / 255.0
        img_tensor = np.transpose(img_array, (2, 0, 1))[np.newaxis, ...]
        self.idx += 1
        return {self.input_name: img_tensor}

with open("/workspace/train_capped.txt") as f:
    all_train_images = [line.strip() for line in f if line.strip()]
calib_images = all_train_images[:100]
print(f"Using {len(calib_images)} calibration images")

# Shape inference
yolo11n_fp32_prep = yolo11n_fp32_path.replace(".onnx", "_prep.onnx")
model_onnx = onnx.load(yolo11n_fp32_path)
model_onnx = shape_inference.infer_shapes(model_onnx)
onnx.save(model_onnx, yolo11n_fp32_prep)
print("Shape inference done")

# Dynamically find detection head nodes (last 2 numbered model layers)
layer_nums = set()
for node in model_onnx.graph.node:
    m = re.search(r'model\.(\d+)', node.name)
    if m:
        layer_nums.add(int(m.group(1)))
sorted_layers = sorted(layer_nums)
head_layer_nums = sorted_layers[-2:]
head_tags = [f"model.{n}" for n in head_layer_nums]

head_nodes = []
for node in model_onnx.graph.node:
    if any(tag in node.name for tag in head_tags):
        head_nodes.append(node.name)
print(f"Detection head layers: {head_tags}")
print(f"Excluding {len(head_nodes)} head nodes from quantization")

# Static quantization
sess = ort.InferenceSession(yolo11n_fp32_prep)
input_name = sess.get_inputs()[0].name
reader = ImageCalibrationReader(calib_images, input_name, imgsz=832)
yolo11n_int8_path = os.path.join(OUTPUT_DIR, "yolo11n_int8.onnx")

quantize_static(
    yolo11n_fp32_prep, yolo11n_int8_path,
    calibration_data_reader=reader,
    quant_format=QuantFormat.QDQ, per_channel=True,
    weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8,
    nodes_to_exclude=head_nodes,
)

print(f"\nINT8 ONNX saved to: {yolo11n_int8_path}")
print(f"FP32 size: {os.path.getsize(yolo11n_fp32_path) / (1024*1024):.1f} MB")
print(f"INT8 size: {os.path.getsize(yolo11n_int8_path) / (1024*1024):.1f} MB")
os.remove(yolo11n_fp32_prep)

Using 100 calibration images
Shape inference done
Detection head layers: ['model.22', 'model.23']
Excluding 116 head nodes from quantization



INT8 ONNX saved to: /workspace/FYP_RESULTS/Final-Year-Project/runs/quantized/yolo11n_int8.onnx
FP32 size: 10.2 MB
INT8 size: 5.4 MB


In [10]:
# Evaluate INT8 accuracy
int8_model_11n = YOLO(yolo11n_int8_path)
metrics_11n_int8 = int8_model_11n.val(data=YOLO_DATA, split="test", imgsz=832)

print(f"\n===== YOLO11n INT8 ONNX — TEST METRICS =====")
print(f"mAP@0.50:        {metrics_11n_int8.box.map50:.4f}")
print(f"mAP@0.50:0.95:   {metrics_11n_int8.box.map:.4f}")
print(f"Precision:        {metrics_11n_int8.box.mp:.4f}")
print(f"Recall:           {metrics_11n_int8.box.mr:.4f}")
print(f"==============================================")

print(f"\n--- Quantization Impact ---")
print(f"mAP@50-95 drop:  {metrics_11n_fp32.box.map - metrics_11n_int8.box.map:+.4f}")
print(f"Precision drop:  {metrics_11n_fp32.box.mp - metrics_11n_int8.box.mp:+.4f}")
print(f"Recall drop:     {metrics_11n_fp32.box.mr - metrics_11n_int8.box.mr:+.4f}")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Ultralytics 8.4.16 🚀 Python-3.12.3 torch-2.9.1+cu128 CPU (Intel Xeon Gold 6133 CPU @ 2.50GHz)
Loading /workspace/FYP_RESULTS/Final-Year-Project/runs/quantized/yolo11n_int8.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.2 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 832, 832)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1164.4±259.4 MB/s, size: 212.5 KB)
val: Scanning /workspace/FYP_RESULTS/Final-Year-Project/datasets/my_data/labels.cache... 660 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 660/660 125.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 660/660 3.7it/s 2:58<0.2s
                   all        660        793      0.917      0.839      0.906      0.699
Speed: 2.1ms preprocess, 225.2ms inferen

---
## Part 1c: YOLOv10n — Export & Quantize

In [4]:
from ultralytics import YOLO

# Load trained YOLOv10n PyTorch model
model_v10n = YOLO(YOLOV10N_PT)

# Export to FP32 ONNX (end2end=False for quantization-friendly graph)
onnx_path = model_v10n.export(format="onnx", imgsz=832, end2end=False)

# Copy to output directory with clean name
yolov10n_fp32_path = os.path.join(OUTPUT_DIR, "yolov10n_fp32.onnx")
shutil.copy2(onnx_path, yolov10n_fp32_path)
print(f"\nFP32 ONNX saved to: {yolov10n_fp32_path}")
print(f"Size: {os.path.getsize(yolov10n_fp32_path) / (1024*1024):.1f} MB")

Ultralytics 8.4.16 🚀 Python-3.12.3 torch-2.9.1+cu128 CPU (Intel Xeon Gold 6133 CPU @ 2.50GHz)


YOLOv10n summary (fused): 125 layers, 2,694,806 parameters, 0 gradients, 6.5 GFLOPs

PyTorch: starting from '/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolov10n_real_e402/weights/best.pt' with input shape (1, 3, 832, 832) BCHW and output shape(s) (1, 5, 14196) (5.5 MB)

ONNX: starting export with onnx 1.19.1 opset 22...


/usr/local/lib/python3.12/dist-packages/torch/onnx/_internal/torchscript_exporter/utils.py:1447: OnnxExporterWarning: Exporting to ONNX opset version 22 is not supported. by 'torch.onnx.export()'. The highest opset version supported is 20. To use a newer opset version, consider 'torch.onnx.export(..., dynamo=True)'. 
  warnings.warn(


ONNX: slimming with onnxslim 0.1.85...
ONNX: export success ✅ 1.6s, saved as '/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolov10n_real_e402/weights/best.onnx' (9.0 MB)

Export complete (2.1s)
Results saved to /workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolov10n_real_e402/weights
Predict:         yolo predict task=detect model=/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolov10n_real_e402/weights/best.onnx imgsz=832 
Validate:        yolo val task=detect model=/workspace/FYP_RESULTS/Final-Year-Project/runs/detect/yolov10n_real_e402/weights/best.onnx imgsz=832 data=mydata_capped.yaml  
Visualize:       https://netron.app

FP32 ONNX saved to: /workspace/FYP_RESULTS/Final-Year-Project/runs/quantized/yolov10n_fp32.onnx
Size: 9.0 MB


In [5]:
# Verify FP32 ONNX accuracy matches PyTorch
onnx_model_v10n = YOLO(yolov10n_fp32_path)
metrics_v10n_fp32 = onnx_model_v10n.val(data=YOLO_DATA, split="test", imgsz=832)

print(f"\n===== YOLOv10n FP32 ONNX — TEST METRICS =====")
print(f"mAP@0.50:        {metrics_v10n_fp32.box.map50:.4f}")
print(f"mAP@0.50:0.95:   {metrics_v10n_fp32.box.map:.4f}")
print(f"Precision:        {metrics_v10n_fp32.box.mp:.4f}")
print(f"Recall:           {metrics_v10n_fp32.box.mr:.4f}")
print(f"==============================================")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Ultralytics 8.4.16 🚀 Python-3.12.3 torch-2.9.1+cu128 CPU (Intel Xeon Gold 6133 CPU @ 2.50GHz)
Loading /workspace/FYP_RESULTS/Final-Year-Project/runs/quantized/yolov10n_fp32.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.2 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 832, 832)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2721.7±620.8 MB/s, size: 247.3 KB)
val: Scanning /workspace/FYP_RESULTS/Final-Year-Project/datasets/my_data/labels... 660 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 660/660 1.4Kit/s 0.5s0.1ss
val: New cache created: /workspace/FYP_RESULTS/Final-Year-Project/datasets/my_data/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 660/660 2.7it/s 4:08<0.4s
                   all        660  

In [6]:
import numpy as np
import re
import onnxruntime as ort
import onnx
from onnx import shape_inference
from PIL import Image
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantFormat, QuantType

class ImageCalibrationReader(CalibrationDataReader):
    def __init__(self, image_paths, input_name, imgsz=832):
        self.images = image_paths
        self.input_name = input_name
        self.imgsz = imgsz
        self.idx = 0
    def get_next(self):
        if self.idx >= len(self.images):
            return None
        img = Image.open(self.images[self.idx]).convert("RGB")
        img = img.resize((self.imgsz, self.imgsz))
        img_array = np.array(img, dtype=np.float32) / 255.0
        img_tensor = np.transpose(img_array, (2, 0, 1))[np.newaxis, ...]
        self.idx += 1
        return {self.input_name: img_tensor}

with open("/workspace/train_capped.txt") as f:
    all_train_images = [line.strip() for line in f if line.strip()]
calib_images = all_train_images[:100]
print(f"Using {len(calib_images)} calibration images")

# Shape inference
yolov10n_fp32_prep = yolov10n_fp32_path.replace(".onnx", "_prep.onnx")
model_onnx = onnx.load(yolov10n_fp32_path)
model_onnx = shape_inference.infer_shapes(model_onnx)
onnx.save(model_onnx, yolov10n_fp32_prep)
print("Shape inference done")

# Dynamically find detection head nodes (last 2 numbered model layers)
layer_nums = set()
for node in model_onnx.graph.node:
    m = re.search(r'model\.(\d+)', node.name)
    if m:
        layer_nums.add(int(m.group(1)))
sorted_layers = sorted(layer_nums)
head_layer_nums = sorted_layers[-2:]
head_tags = [f"model.{n}" for n in head_layer_nums]

head_nodes = []
for node in model_onnx.graph.node:
    if any(tag in node.name for tag in head_tags):
        head_nodes.append(node.name)
print(f"Detection head layers: {head_tags}")
print(f"Excluding {len(head_nodes)} head nodes from quantization")

# Static quantization
sess = ort.InferenceSession(yolov10n_fp32_prep)
input_name = sess.get_inputs()[0].name
reader = ImageCalibrationReader(calib_images, input_name, imgsz=832)
yolov10n_int8_path = os.path.join(OUTPUT_DIR, "yolov10n_int8.onnx")

quantize_static(
    yolov10n_fp32_prep, yolov10n_int8_path,
    calibration_data_reader=reader,
    quant_format=QuantFormat.QDQ, per_channel=True,
    weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8,
    nodes_to_exclude=head_nodes,
)

print(f"\nINT8 ONNX saved to: {yolov10n_int8_path}")
print(f"FP32 size: {os.path.getsize(yolov10n_fp32_path) / (1024*1024):.1f} MB")
print(f"INT8 size: {os.path.getsize(yolov10n_int8_path) / (1024*1024):.1f} MB")
os.remove(yolov10n_fp32_prep)

Using 100 calibration images
Shape inference done
Detection head layers: ['model.22', 'model.23']
Excluding 108 head nodes from quantization



INT8 ONNX saved to: /workspace/FYP_RESULTS/Final-Year-Project/runs/quantized/yolov10n_int8.onnx
FP32 size: 9.0 MB
INT8 size: 4.8 MB


In [7]:
# Evaluate INT8 accuracy
int8_model_v10n = YOLO(yolov10n_int8_path)
metrics_v10n_int8 = int8_model_v10n.val(data=YOLO_DATA, split="test", imgsz=832)

print(f"\n===== YOLOv10n INT8 ONNX — TEST METRICS =====")
print(f"mAP@0.50:        {metrics_v10n_int8.box.map50:.4f}")
print(f"mAP@0.50:0.95:   {metrics_v10n_int8.box.map:.4f}")
print(f"Precision:        {metrics_v10n_int8.box.mp:.4f}")
print(f"Recall:           {metrics_v10n_int8.box.mr:.4f}")
print(f"==============================================")

print(f"\n--- Quantization Impact ---")
print(f"mAP@50-95 drop:  {metrics_v10n_fp32.box.map - metrics_v10n_int8.box.map:+.4f}")
print(f"Precision drop:  {metrics_v10n_fp32.box.mp - metrics_v10n_int8.box.mp:+.4f}")
print(f"Recall drop:     {metrics_v10n_fp32.box.mr - metrics_v10n_int8.box.mr:+.4f}")

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify','pose' or 'obb'.
Ultralytics 8.4.16 🚀 Python-3.12.3 torch-2.9.1+cu128 CPU (Intel Xeon Gold 6133 CPU @ 2.50GHz)
Loading /workspace/FYP_RESULTS/Final-Year-Project/runs/quantized/yolov10n_int8.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.24.2 with CPUExecutionProvider
Setting batch=1 input of shape (1, 3, 832, 832)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2177.9±379.1 MB/s, size: 194.8 KB)
val: Scanning /workspace/FYP_RESULTS/Final-Year-Project/datasets/my_data/labels.cache... 660 images, 23 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 660/660 76.9Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 660/660 3.3it/s 3:22<0.3s
                   all        660        793      0.958      0.881      0.917        0.7
Speed: 3.4ms preprocess, 258.3ms inferen

---
## Summary: Accuracy, File Sizes & Raspberry Pi 5 Latency

In [17]:
import os

print("=" * 75)
print("EXPERIMENT 1: Real vs Synthetic Training Data (YOLO11n)")
print("=" * 75)
print(f"{'Model':<35} {'mAP@50':>8} {'mAP@50-95':>10} {'Precision':>10} {'Recall':>8}")
print("-" * 73)
for name, m50, m, p, r in [
    ("YOLO11n COCO pretrained",     0.853, 0.671, 0.922, 0.753),
    ("YOLO11n Real fine-tuned",     0.931, 0.735, 0.949, 0.893),
    ("YOLO11n Synthetic fine-tuned", 0.719, 0.503, 0.769, 0.615),
]:
    print(f"{name:<35} {m50:>8.3f} {m:>10.3f} {p:>10.3f} {r:>8.3f}")

print("\n" + "=" * 75)
print("EXPERIMENT 2: Architecture Comparison (fine-tuned on real data)")
print("=" * 75)
print(f"{'Model':<15} {'mAP@50':>8} {'mAP@50-95':>10} {'Precision':>10} {'Recall':>8}")
print("-" * 53)
for name, m50, m, p, r in [
    ("YOLO11n",   0.931, 0.735, 0.949, 0.892),
    ("YOLOv10n",  0.935, 0.702, 0.960, 0.859),
    ("YOLO26n",   0.924, 0.745, 0.931, 0.884),
    ("RF-DETR-N", 0.916, 0.724, None,  0.770),
]:
    p_str = f"{p:>10.3f}" if p else f"{'N/A':>10}"
    print(f"{name:<15} {m50:>8.3f} {m:>10.3f} {p_str} {r:>8.3f}")

print("\n" + "=" * 75)
print("EXPERIMENT 3: ONNX Export & INT8 Quantization")
print("=" * 75)

# YOLO11n
print("\n--- YOLO11n ---")
print(f"{'Format':<16} {'mAP@50':>8} {'mAP@50-95':>10} {'Precision':>10} {'Recall':>8} {'Size (MB)':>10}")
print("-" * 64)
yolo11n_fp32_sz = os.path.getsize(os.path.join(OUTPUT_DIR, "yolo11n_fp32.onnx")) / (1024*1024)
yolo11n_int8_sz = os.path.getsize(os.path.join(OUTPUT_DIR, "yolo11n_int8.onnx")) / (1024*1024)
print(f"{'FP32 ONNX':<16} {metrics_11n_fp32.box.map50:>8.3f} {metrics_11n_fp32.box.map:>10.3f} {metrics_11n_fp32.box.mp:>10.3f} {metrics_11n_fp32.box.mr:>8.3f} {yolo11n_fp32_sz:>10.1f}")
print(f"{'INT8 ONNX':<16} {metrics_11n_int8.box.map50:>8.3f} {metrics_11n_int8.box.map:>10.3f} {metrics_11n_int8.box.mp:>10.3f} {metrics_11n_int8.box.mr:>8.3f} {yolo11n_int8_sz:>10.1f}")
drop_pct_11n = (1 - yolo11n_int8_sz / yolo11n_fp32_sz) * 100
print(f"{'Drop':<16} {metrics_11n_fp32.box.map50 - metrics_11n_int8.box.map50:>+8.3f} {metrics_11n_fp32.box.map - metrics_11n_int8.box.map:>+10.3f} {metrics_11n_fp32.box.mp - metrics_11n_int8.box.mp:>+10.3f} {metrics_11n_fp32.box.mr - metrics_11n_int8.box.mr:>+8.3f} {f'-{drop_pct_11n:.0f}%':>10}")

# YOLOv10n
print("\n--- YOLOv10n ---")
print(f"{'Format':<16} {'mAP@50':>8} {'mAP@50-95':>10} {'Precision':>10} {'Recall':>8} {'Size (MB)':>10}")
print("-" * 64)
yolov10n_fp32_sz = os.path.getsize(os.path.join(OUTPUT_DIR, "yolov10n_fp32.onnx")) / (1024*1024)
yolov10n_int8_sz = os.path.getsize(os.path.join(OUTPUT_DIR, "yolov10n_int8.onnx")) / (1024*1024)
print(f"{'FP32 ONNX':<16} {metrics_v10n_fp32.box.map50:>8.3f} {metrics_v10n_fp32.box.map:>10.3f} {metrics_v10n_fp32.box.mp:>10.3f} {metrics_v10n_fp32.box.mr:>8.3f} {yolov10n_fp32_sz:>10.1f}")
print(f"{'INT8 ONNX':<16} {metrics_v10n_int8.box.map50:>8.3f} {metrics_v10n_int8.box.map:>10.3f} {metrics_v10n_int8.box.mp:>10.3f} {metrics_v10n_int8.box.mr:>8.3f} {yolov10n_int8_sz:>10.1f}")
drop_pct_v10n = (1 - yolov10n_int8_sz / yolov10n_fp32_sz) * 100
print(f"{'Drop':<16} {metrics_v10n_fp32.box.map50 - metrics_v10n_int8.box.map50:>+8.3f} {metrics_v10n_fp32.box.map - metrics_v10n_int8.box.map:>+10.3f} {metrics_v10n_fp32.box.mp - metrics_v10n_int8.box.mp:>+10.3f} {metrics_v10n_fp32.box.mr - metrics_v10n_int8.box.mr:>+8.3f} {f'-{drop_pct_v10n:.0f}%':>10}")

# YOLO26n
print("\n--- YOLO26n ---")
print(f"{'Format':<16} {'mAP@50':>8} {'mAP@50-95':>10} {'Precision':>10} {'Recall':>8} {'Size (MB)':>10}")
print("-" * 64)
yolo26n_fp32_sz = os.path.getsize(os.path.join(OUTPUT_DIR, "yolo26n_fp32.onnx")) / (1024*1024)
yolo26n_int8_sz = os.path.getsize(os.path.join(OUTPUT_DIR, "yolo26n_int8.onnx")) / (1024*1024)
print(f"{'FP32 ONNX':<16} {metrics_fp32.box.map50:>8.3f} {metrics_fp32.box.map:>10.3f} {metrics_fp32.box.mp:>10.3f} {metrics_fp32.box.mr:>8.3f} {yolo26n_fp32_sz:>10.1f}")
print(f"{'INT8 ONNX':<16} {metrics_int8.box.map50:>8.3f} {metrics_int8.box.map:>10.3f} {metrics_int8.box.mp:>10.3f} {metrics_int8.box.mr:>8.3f} {yolo26n_int8_sz:>10.1f}")
drop_pct_26n = (1 - yolo26n_int8_sz / yolo26n_fp32_sz) * 100
print(f"{'Drop':<16} {metrics_fp32.box.map50 - metrics_int8.box.map50:>+8.3f} {metrics_fp32.box.map - metrics_int8.box.map:>+10.3f} {metrics_fp32.box.mp - metrics_int8.box.mp:>+10.3f} {metrics_fp32.box.mr - metrics_int8.box.mr:>+8.3f} {f'-{drop_pct_26n:.0f}%':>10}")

# ── Raspberry Pi 5 Latency Results ──
print("\n" + "=" * 75)
print("RASPBERRY PI 5 LATENCY (ONNX Runtime, CPU, 200 images, 832x832)")
print("=" * 75)
print(f"{'Model':<25} {'Size (MB)':>10} {'Avg (ms)':>10} {'Median (ms)':>12} {'FPS':>8} {'Speedup':>9}")
print("-" * 76)

pi5_results = [
    ("YOLO11n FP32",  10.2, 328.7, 321.8, 3.04,  None),
    ("YOLO11n INT8",   5.4, 207.5, 205.5, 4.82,  328.7/207.5),
    ("YOLOv10n FP32",  9.0, 385.8, 388.6, 2.59,  None),
    ("YOLOv10n INT8",  4.8, 246.8, 247.4, 4.05,  385.8/246.8),
    ("YOLO26n FP32",   9.5, 333.9, 346.8, 3.00,  None),
    ("YOLO26n INT8",   4.6, 188.1, 199.3, 5.32,  333.9/188.1),
]

for name, size, avg, median, fps, speedup in pi5_results:
    sp_str = f"{speedup:.2f}x" if speedup else "—"
    print(f"{name:<25} {size:>10.1f} {avg:>10.1f} {median:>12.1f} {fps:>8.2f} {sp_str:>9}")

print("-" * 76)
print("Best: YOLO26n INT8 — 5.32 FPS, 188.1 ms avg, 4.6 MB")

print("\n--- Exported Files ---")
print(f"{'File':<35} {'Size (MB)':>10}")
print("-" * 47)
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    if os.path.isfile(fpath) and 'rfdetr' not in fname:
        size = os.path.getsize(fpath) / (1024 * 1024)
        print(f"{fname:<35} {size:>10.1f}")

EXPERIMENT 1: Real vs Synthetic Training Data (YOLO11n)
Model                                 mAP@50  mAP@50-95  Precision   Recall
-------------------------------------------------------------------------
YOLO11n COCO pretrained                0.853      0.671      0.923    0.752
YOLO11n Real fine-tuned                0.931      0.735      0.949    0.892
YOLO11n Synthetic fine-tuned           0.720      0.477      0.872    0.607

EXPERIMENT 2: Architecture Comparison (fine-tuned on real data)
Model             mAP@50  mAP@50-95  Precision   Recall
-----------------------------------------------------
YOLO11n            0.931      0.735      0.949    0.892
YOLOv10n           0.935      0.702      0.960    0.859
YOLO26n            0.924      0.745      0.931    0.884
RF-DETR-N          0.896      0.708        N/A    0.768

EXPERIMENT 3: ONNX Export & INT8 Quantization

--- YOLO11n ---
Format             mAP@50  mAP@50-95  Precision   Recall  Size (MB)
-----------------------------------

In [16]:
# Copy ONNX models to RP5 folder for Raspberry Pi 5 benchmarking
import shutil

rp5_dir = "/workspace/FYP_RESULTS/Final-Year-Project/RP5"

for model_name in ["yolo11n_fp32.onnx", "yolo11n_int8.onnx",
                   "yolov10n_fp32.onnx", "yolov10n_int8.onnx",
                   "yolo26n_fp32.onnx", "yolo26n_int8.onnx"]:
    src = os.path.join(OUTPUT_DIR, model_name)
    dst = os.path.join(rp5_dir, model_name)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f"Copied {model_name} ({os.path.getsize(dst) / (1024*1024):.1f} MB)")
    else:
        print(f"WARNING: {model_name} not found — run export/quantize cells first")

print(f"\n--- RP5 Folder Contents ---")
for fname in sorted(os.listdir(rp5_dir)):
    fpath = os.path.join(rp5_dir, fname)
    if os.path.isfile(fpath):
        size = os.path.getsize(fpath) / (1024 * 1024)
        print(f"  {fname:<35} {size:>10.1f} MB")

Copied yolo11n_fp32.onnx (10.2 MB)
Copied yolo11n_int8.onnx (5.4 MB)
Copied yolov10n_fp32.onnx (9.0 MB)
Copied yolov10n_int8.onnx (4.8 MB)
Copied yolo26n_fp32.onnx (9.5 MB)
Copied yolo26n_int8.onnx (4.6 MB)

--- RP5 Folder Contents ---
  yolo11n_fp32.onnx                         10.2 MB
  yolo11n_int8.onnx                          5.4 MB
  yolo26n_fp32.onnx                          9.5 MB
  yolo26n_int8.onnx                          4.6 MB
  yolov10n_fp32.onnx                         9.0 MB
  yolov10n_int8.onnx                         4.8 MB


---
## Report Tables

Copy-paste ready tables for the FYP report.

In [18]:
# ================================================================
# TABLE 1: Experiment 2 — Architecture Comparison (4 Models)
# ================================================================
print("TABLE 1: Architecture Comparison — All 4 Models")
print("=" * 70)
print(f"{'Model':<15} {'mAP@50':>8} {'mAP@50-95':>11} {'Precision':>11} {'Recall':>8}")
print("-" * 55)
for name, m50, m, p, r in [
    ("YOLO11n",   0.931, 0.735, 0.949, 0.892),
    ("YOLOv10n",  0.935, 0.702, 0.960, 0.859),
    ("YOLO26n",   0.924, 0.745, 0.931, 0.884),
    ("RF-DETR-N", 0.916, 0.724, None,  0.770),
]:
    p_str = f"{p:.3f}" if p else "N/A"
    print(f"{name:<15} {m50:>8.3f} {m:>11.3f} {p_str:>11} {r:>8.3f}")

# ================================================================
# TABLE 2: Experiment 2 — Architecture Comparison (3 YOLO Models)
# ================================================================
print("\n\nTABLE 2: Architecture Comparison — 3 YOLO Models")
print("=" * 70)
print(f"{'Model':<15} {'mAP@50':>8} {'mAP@50-95':>11} {'Precision':>11} {'Recall':>8}")
print("-" * 55)
for name, m50, m, p, r in [
    ("YOLO11n",  0.931, 0.735, 0.949, 0.892),
    ("YOLOv10n", 0.935, 0.702, 0.960, 0.859),
    ("YOLO26n",  0.924, 0.745, 0.931, 0.884),
]:
    print(f"{name:<15} {m50:>8.3f} {m:>11.3f} {p:>11.3f} {r:>8.3f}")

TABLE 1: Architecture Comparison — All 4 Models
Model             mAP@50   mAP@50-95   Precision   Recall
-------------------------------------------------------
YOLO11n            0.931       0.735       0.949    0.892
YOLOv10n           0.935       0.702       0.960    0.859
YOLO26n            0.924       0.745       0.931    0.884
RF-DETR-N          0.896       0.708         N/A    0.768


TABLE 2: Architecture Comparison — 3 YOLO Models
Model             mAP@50   mAP@50-95   Precision   Recall
-------------------------------------------------------
YOLO11n            0.931       0.735       0.949    0.892
YOLOv10n           0.935       0.702       0.960    0.859
YOLO26n            0.924       0.745       0.931    0.884


In [19]:
# ================================================================
# TABLE 3: Experiment 3 — Accuracy Pipeline (PyTorch → FP32 → INT8)
# ================================================================
print("TABLE 3: Accuracy Through Export & Quantization Pipeline")
print("=" * 85)
print(f"{'Model':<12} {'':>10} {'PyTorch':>10} {'FP32 ONNX':>11} {'ONNX Drop':>11} {'INT8 ONNX':>11} {'INT8 Drop':>11}")
print("-" * 78)

data = [
    ("YOLO11n",  0.931, 0.921, 0.906, 0.735, 0.723, 0.699),
    ("YOLOv10n", 0.935, 0.926, 0.917, 0.702, 0.688, 0.700),
    ("YOLO26n",  0.924, 0.923, 0.907, 0.745, 0.742, 0.712),
]

for name, pt50, fp50, i50, pt95, fp95, i95 in data:
    print(f"{name:<12} {'mAP@50':>10} {pt50:>10.3f} {fp50:>11.3f} {fp50-pt50:>+11.3f} {i50:>11.3f} {i50-pt50:>+11.3f}")
    print(f"{'':12} {'mAP@50-95':>10} {pt95:>10.3f} {fp95:>11.3f} {fp95-pt95:>+11.3f} {i95:>11.3f} {i95-pt95:>+11.3f}")
    print()

TABLE 3: Accuracy Through Export & Quantization Pipeline
Model                      PyTorch   FP32 ONNX   ONNX Drop   INT8 ONNX   INT8 Drop
------------------------------------------------------------------------------
YOLO11n          mAP@50      0.931       0.921      -0.010       0.906      -0.025
              mAP@50-95      0.735       0.723      -0.012       0.699      -0.036

YOLOv10n         mAP@50      0.935       0.926      -0.009       0.917      -0.018
              mAP@50-95      0.702       0.688      -0.014       0.700      -0.002

YOLO26n          mAP@50      0.924       0.923      -0.001       0.907      -0.017
              mAP@50-95      0.745       0.742      -0.003       0.712      -0.033



In [20]:
# ================================================================
# TABLE 4: Experiment 3 — Model Size Comparison
# ================================================================
print("TABLE 4: ONNX Model Size — FP32 vs INT8")
print("=" * 60)
print(f"{'Model':<12} {'FP32 (MB)':>11} {'INT8 (MB)':>11} {'Reduction':>11} {'Reduction %':>13}")
print("-" * 60)

sizes = [
    ("YOLO11n",  10.2, 5.4),
    ("YOLOv10n",  9.0, 4.8),
    ("YOLO26n",   9.5, 4.6),
]

for name, fp32, int8 in sizes:
    reduction = fp32 - int8
    pct = (1 - int8 / fp32) * 100
    print(f"{name:<12} {fp32:>11.1f} {int8:>11.1f} {reduction:>+11.1f} {pct:>12.0f}%")

TABLE 4: ONNX Model Size — FP32 vs INT8
Model          FP32 (MB)   INT8 (MB)   Reduction   Reduction %
------------------------------------------------------------
YOLO11n             10.2         5.4        +4.8           47%
YOLOv10n             9.0         4.8        +4.2           47%
YOLO26n              9.5         4.6        +4.9           52%


In [21]:
# ================================================================
# TABLE 5: Experiment 3 — Raspberry Pi 5 Latency & FPS
# ================================================================
print("TABLE 5: Raspberry Pi 5 Inference Performance")
print("=" * 80)
print(f"{'Model':<20} {'Format':>7} {'Avg (ms)':>10} {'Median (ms)':>12} {'Std (ms)':>10} {'FPS':>8} {'Speedup':>9}")
print("-" * 78)

pi5 = [
    ("YOLO11n",  "FP32", 328.7, 321.8, 20.0, 3.04),
    ("YOLO11n",  "INT8", 207.5, 205.5,  9.7, 4.82),
    ("YOLOv10n", "FP32", 385.8, 388.6, 34.1, 2.59),
    ("YOLOv10n", "INT8", 246.8, 247.4, 19.0, 4.05),
    ("YOLO26n",  "FP32", 333.9, 346.8, 22.2, 3.00),
    ("YOLO26n",  "INT8", 188.1, 199.3, 16.6, 5.32),
]

fp32_latencies = {}
for name, fmt, avg, med, std, fps in pi5:
    if fmt == "FP32":
        fp32_latencies[name] = avg
        sp_str = "—"
    else:
        speedup = fp32_latencies[name] / avg
        sp_str = f"{speedup:.2f}x"
    print(f"{name:<20} {fmt:>7} {avg:>10.1f} {med:>12.1f} {std:>10.1f} {fps:>8.2f} {sp_str:>9}")

print("-" * 78)
print("Device: Raspberry Pi 5 | Provider: CPUExecutionProvider | Images: 200 | Resolution: 832x832")

TABLE 5: Raspberry Pi 5 Inference Performance
Model                 Format   Avg (ms)  Median (ms)   Std (ms)      FPS   Speedup
------------------------------------------------------------------------------
YOLO11n                 FP32      328.7        321.8       20.0     3.04         —
YOLO11n                 INT8      207.5        205.5        9.7     4.82     1.58x
YOLOv10n                FP32      385.8        388.6       34.1     2.59         —
YOLOv10n                INT8      246.8        247.4       19.0     4.05     1.56x
YOLO26n                 FP32      333.9        346.8       22.2     3.00         —
YOLO26n                 INT8      188.1        199.3       16.6     5.32     1.78x
------------------------------------------------------------------------------
Device: Raspberry Pi 5 | Provider: CPUExecutionProvider | Images: 200 | Resolution: 832x832
